# 02 – Build Master Dataset
Reads raw files, cleans each source, merges into quarterly and annual panels.

**Outputs:** `data/processed/quarterly_master.csv`, `data/processed/annual_net_additions.csv`

In [49]:
import pandas as pd
import numpy as np
import os

In [50]:
# 1. Starts & Completions (quarterly, England)
df_sc = pd.read_excel(
    "../data/raw/ons_starts_completions_england.xlsx",
    sheet_name="1b",
    skiprows=5
)
df_sc = df_sc.dropna(subset=["Period", "Started - All Dwellings"])

df_sc["year"] = df_sc["Period"].str.extract(r"(\d{4})").astype(int)
df_sc["quarter"] = df_sc["Period"].apply(
    lambda x: 1 if "Jan" in str(x) else 2 if "Apr" in str(x) else 3 if "Jul" in str(x) else 4
)
df_sc["date"] = pd.to_datetime(
    df_sc["year"].astype(str) + "-" + (df_sc["quarter"] * 3 - 2).astype(str) + "-01"
)

df_sc = df_sc[["date", "year", "quarter",
               "Started - All Dwellings", "Started - Private Enterprise",
               "Completed - All Dwellings", "Completed - Private Enterprise"]].copy()
df_sc.columns = ["date", "year", "quarter",
                 "starts_all", "starts_private", "comp_all", "comp_private"]

print(f"Starts/completions: {df_sc.shape[0]} quarters, {df_sc.date.min().date()} to {df_sc.date.max().date()}")

Starts/completions: 191 quarters, 1978-01-01 to 2025-07-01


In [51]:
# 2. OBR Housing Market (quarterly, UK)
df_obr = pd.read_excel(
    "../data/raw/obr_economy_march2026.xlsx",
    sheet_name="1.16",
    skiprows=1
)
df_obr = df_obr.iloc[:, 1:]  # drop empty first column

df_obr.columns = ["quarter_label", "hpi", "hpi_yoy", "transactions",
                   "starts_uk", "comp_uk", "housing_stock",
                   "net_additions_uk", "turnover_rate"]
df_obr = df_obr.iloc[1:].copy()  # drop header row

# Keep only quarterly rows (e.g. '2008Q1'), drop annual/fiscal-year/footnotes
mask = df_obr["quarter_label"].astype(str).str.match(r"^\d{4}Q\d$")
df_obr = df_obr[mask].copy()

df_obr["year"] = df_obr["quarter_label"].str[:4].astype(int)
df_obr["quarter"] = df_obr["quarter_label"].str[-1].astype(int)
df_obr["date"] = pd.to_datetime(
    df_obr["year"].astype(str) + "-" + (df_obr["quarter"] * 3 - 2).astype(str) + "-01"
)

num_cols = ["hpi", "hpi_yoy", "transactions", "starts_uk", "comp_uk",
            "housing_stock", "net_additions_uk", "turnover_rate"]
df_obr[num_cols] = df_obr[num_cols].apply(pd.to_numeric, errors="coerce")
df_obr = df_obr[["date", "year", "quarter"] + num_cols].reset_index(drop=True)

print(f"OBR housing market: {df_obr.shape[0]} quarters, {df_obr.date.min().date()} to {df_obr.date.max().date()}")
print(f"  Note: includes OBR forecast quarters from 2025 onwards")

OBR housing market: 93 quarters, 2008-01-01 to 2031-01-01
  Note: includes OBR forecast quarters from 2025 onwards


In [52]:
# 3. Net Additions (Yearly, England)
df_na = pd.read_excel(
    "../data/raw/mhclg_net_additions_england.ods",
    sheet_name="LT120_unrounded",
    engine="odf"
)

# Years across columns, components down rows — extract what we need
years_raw = df_na.iloc[3, 1:-2].tolist()
years = [str(y).split(" ")[0] for y in years_raw]
labels = df_na.iloc[:, 0].tolist()

rows_we_want = {
    "new_build_comp":      "New build completions",
    "net_conversions":     "Net conversions",
    "net_change_of_use":   "Net change of use",
    "net_other_gains":     "Net other gains",
    "demolitions":         "Demolitions",
    "census_adjustments":  "Census adjustments",
    "total_net_additions": "Total net additional dwellings",
}

data = {"fiscal_year": years}
for col_name, label in rows_we_want.items():
    row_idx = labels.index(label)
    values = df_na.iloc[row_idx, 1:-2].tolist()
    data[col_name] = pd.to_numeric(values, errors="coerce")

df_net = pd.DataFrame(data)
df_net["year"] = df_net["fiscal_year"].str[:4].astype(int)

print(f"Net additions: {df_net.shape[0]} years, {df_net.year.min()} to {df_net.year.max()}")
print(f"  Demolitions fell from {df_net.demolitions.iloc[0]:.0f} (2006-07) to {df_net.demolitions.iloc[-1]:.0f} (2024-25)")

Net additions: 19 years, 2006 to 2024
  Demolitions fell from 22290 (2006-07) to 4632 (2024-25)


In [53]:
# 4. EPC New builds (Quarterly, England)
df_epc = pd.read_excel(
    "../data/raw/epc_new_builds.ods",
    sheet_name="NB1_England_Only",
    engine="odf"
)

df_epc.columns = df_epc.iloc[2].tolist()
df_epc = df_epc.iloc[3:].copy()
df_epc = df_epc.dropna(subset=["Quarter"]).copy()  # keep quarterly rows only

df_epc["year"] = df_epc["Year"].astype(int)
df_epc["quarter"] = df_epc["Quarter"].str.split("/").str[1].astype(int)
df_epc["date"] = pd.to_datetime(
    df_epc["year"].astype(str) + "-" + (df_epc["quarter"] * 3 - 2).astype(str) + "-01"
)

df_epc = df_epc[["date", "year", "quarter", "Number Lodgements"]].copy()
df_epc.columns = ["date", "year", "quarter", "epc_lodgements"]
df_epc["epc_lodgements"] = pd.to_numeric(df_epc["epc_lodgements"], errors="coerce")
df_epc = df_epc.reset_index(drop=True)

print(f"EPC lodgements: {df_epc.shape[0]} quarters, {df_epc.date.min().date()} to {df_epc.date.max().date()}")

EPC lodgements: 69 quarters, 2008-10-01 to 2025-10-01


In [54]:
# 5. Nationwide House Price Index (quarterly, UK, 1952–present)
df_hpi = pd.read_excel(
    "../data/raw/nationwide_housing_prices_1952.xlsx",
    sheet_name="UK HP Since 1952",
    skiprows=4,
    header=None
)

# Keep only columns 0 (label), 1 (index), 2 (price) — All Houses series
df_hpi = df_hpi.iloc[:, :3].copy()
df_hpi.columns = ["quarter_label", "hpi_index", "hpi_price"]

# Labels are formatted "Q4 1952", "Q1 1953" etc.
df_hpi = df_hpi.dropna(subset=["quarter_label"])
df_hpi = df_hpi[df_hpi["quarter_label"].astype(str).str.match(r"^Q\d \d{4}$")].copy()

df_hpi["quarter"] = df_hpi["quarter_label"].str[1].astype(int)
df_hpi["year"] = df_hpi["quarter_label"].str[-4:].astype(int)
df_hpi["date"] = pd.to_datetime(
    df_hpi["year"].astype(str) + "-" + (df_hpi["quarter"] * 3 - 2).astype(str) + "-01"
)
df_hpi["hpi_index"] = pd.to_numeric(df_hpi["hpi_index"], errors="coerce")
df_hpi["hpi_price"] = pd.to_numeric(df_hpi["hpi_price"], errors="coerce")
df_hpi = df_hpi[["date", "year", "quarter", "hpi_index", "hpi_price"]].reset_index(drop=True)

print(f"Nationwide HPI: {df_hpi.shape[0]} quarters, {df_hpi.date.min().date()} to {df_hpi.date.max().date()}")

Nationwide HPI: 294 quarters, 1952-10-01 to 2026-01-01


In [55]:
# 6. Bank of England Base Rate (monthly → quarterly, 1975–present)
df_rate = pd.read_csv("../data/raw/boe_base_rate_1975.csv")
df_rate.columns = ["date_raw", "base_rate"]
df_rate["date_raw"] = pd.to_datetime(df_rate["date_raw"], format="%d %b %y")
df_rate["base_rate"] = pd.to_numeric(df_rate["base_rate"], errors="coerce")

df_rate["year"] = df_rate["date_raw"].dt.year
df_rate["quarter"] = df_rate["date_raw"].dt.quarter
df_rate["date"] = pd.to_datetime(
    df_rate["year"].astype(str) + "-" + (df_rate["quarter"] * 3 - 2).astype(str) + "-01"
)

df_rate = (df_rate.groupby(["date", "year", "quarter"])["base_rate"]
           .mean().reset_index())

print(f"Base rate: {df_rate.shape[0]} quarters, {df_rate.date.min().date()} to {df_rate.date.max().date()}")

Base rate: 205 quarters, 1975-01-01 to 2026-01-01


In [56]:
# 7. Transactions splice: BoE approvals (1987–2004) + HMRC SDLT (2005–present)

# C1. HMRC quarterly residential transactions (seasonally adjusted, UK)
df_hmrc = pd.read_excel(
    "../data/raw/transaction_costs_splice/govuk_transactions_2005.ods",
    sheet_name="Residential_quarterly",
    engine="odf",
    skiprows=4
)
df_hmrc = df_hmrc.iloc[:, [0, -1]].copy()
df_hmrc.columns = ["quarter_label", "transactions_sa"]
df_hmrc = df_hmrc.dropna(subset=["quarter_label"])
df_hmrc = df_hmrc[df_hmrc["quarter_label"].astype(str).str.match(r"^\d{4} Quarter \d$")].copy()

df_hmrc["year"] = df_hmrc["quarter_label"].str[:4].astype(int)
df_hmrc["quarter"] = df_hmrc["quarter_label"].str[-1].astype(int)
df_hmrc["date"] = pd.to_datetime(
    df_hmrc["year"].astype(str) + "-" + (df_hmrc["quarter"] * 3 - 2).astype(str) + "-01"
)
df_hmrc["transactions_sa"] = pd.to_numeric(df_hmrc["transactions_sa"], errors="coerce")
df_hmrc = df_hmrc[["date", "year", "quarter", "transactions_sa"]].reset_index(drop=True)

# C2. BoE mortgage approvals (quarterly, 1987–present) — proxy for pre-2005
df_approvals = pd.read_csv("../data/raw/transaction_costs_splice/boe_mortgage_approvals_1987.csv")
df_approvals.columns = ["date_raw", "approvals"]
df_approvals["date_raw"] = pd.to_datetime(df_approvals["date_raw"], format="%d %b %y")
df_approvals["approvals"] = pd.to_numeric(df_approvals["approvals"], errors="coerce")
df_approvals["year"] = df_approvals["date_raw"].dt.year
df_approvals["quarter"] = df_approvals["date_raw"].dt.quarter
df_approvals["date"] = pd.to_datetime(
    df_approvals["year"].astype(str) + "-" + (df_approvals["quarter"] * 3 - 2).astype(str) + "-01"
)
df_approvals = df_approvals[["date", "year", "quarter", "approvals"]].reset_index(drop=True)

# C3. Splice: scale approvals to HMRC level using 2005–2010 overlap
overlap = df_hmrc[df_hmrc["year"] <= 2010].merge(
    df_approvals, on=["date", "year", "quarter"]
)
scale_factor = overlap["transactions_sa"].mean() / overlap["approvals"].mean()
print(f"Scale factor (approvals → transactions): {scale_factor:.4f}")

df_approvals["transactions_spliced"] = df_approvals["approvals"] * scale_factor

# C4. Combine: approvals (scaled) for pre-2005, HMRC from 2005 onward
df_pre2005 = df_approvals[df_approvals["year"] < 2005][["date", "year", "quarter", "transactions_spliced"]].copy()
df_pre2005.columns = ["date", "year", "quarter", "transactions_sa"]
df_transactions = pd.concat([df_pre2005, df_hmrc], ignore_index=True).sort_values("date")

# Filling in missing 2005 Q1
df_transactions["transactions_sa"] = df_transactions["transactions_sa"].interpolate(method="linear")

# Flag the spliced portion for use as a dummy variable later
df_transactions["transactions_spliced_flag"] = (df_transactions["year"] < 2005).astype(int)

print(f"Spliced transactions: {df_transactions.shape[0]} quarters, {df_transactions.date.min().date()} to {df_transactions.date.max().date()}")
print(f"  Pre-2005 (approvals-based): {df_transactions['transactions_spliced_flag'].sum()} quarters")
print(f"  Post-2005 (HMRC): {(df_transactions['transactions_spliced_flag'] == 0).sum()} quarters")

Scale factor (approvals → transactions): 1.3168
Spliced transactions: 154 quarters, 1987-01-01 to 2025-07-01
  Pre-2005 (approvals-based): 72 quarters
  Post-2005 (HMRC): 82 quarters


In [57]:
# 8. DLUHC dwelling stock (annual → quarterly interpolation)
df_stock = pd.read_excel(
    "../data/raw/govuk_housingstock_annual.ods",
    sheet_name="LT_104",
    engine="odf",
    header=None
)

df_stock = df_stock.iloc[4:].copy()
df_stock.columns = ["date_type", "year", "owner_occ", "private_rent",
                    "social_rent", "LA_rent", "other", "total_stock", "notes"]

df_stock = df_stock[pd.to_numeric(df_stock["year"], errors="coerce").notna()].copy()
df_stock["year"] = df_stock["year"].astype(int)
df_stock["total_stock"] = pd.to_numeric(df_stock["total_stock"], errors="coerce")
df_stock = df_stock[df_stock["year"] >= 1975][["year", "total_stock"]].dropna()

# Deduplicate — where a year has two entries (e.g. 1971 Apr + Dec), keep the last
df_stock = df_stock.groupby("year")["total_stock"].last().reset_index()

df_stock["date"] = pd.to_datetime(df_stock["year"].astype(str) + "-01-01")
df_stock = df_stock.set_index("date")[["total_stock"]]

full_q_index = pd.date_range(start="1975-01-01", end="2025-07-01", freq="QS")
df_stock_q = df_stock.reindex(df_stock.index.union(full_q_index))
df_stock_q = df_stock_q.interpolate(method="cubic")
df_stock_q = df_stock_q.loc[full_q_index].reset_index()
df_stock_q.columns = ["date", "housing_stock_eng"]
df_stock_q["year"] = df_stock_q["date"].dt.year
df_stock_q["quarter"] = df_stock_q["date"].dt.quarter

print(f"Housing stock (interpolated): {df_stock_q.shape[0]} quarters, {df_stock_q.date.min().date()} to {df_stock_q.date.max().date()}")

Housing stock (interpolated): 203 quarters, 1975-01-01 to 2025-07-01


In [58]:
# 9. Merge to quarterly master
df_master = df_sc.copy()
df_master = df_master.merge(df_hpi,          on=["date", "year", "quarter"], how="left")
df_master = df_master.merge(df_rate,         on=["date", "year", "quarter"], how="left")
df_master = df_master.merge(df_transactions, on=["date", "year", "quarter"], how="left")
df_master = df_master.merge(df_stock_q,      on=["date", "year", "quarter"], how="left")
df_master = df_master.merge(df_epc,          on=["date", "year", "quarter"], how="left")
# Drop the OBR columns that are now superseded by better long-run series
df_master = df_master.drop(columns=["hpi", "hpi_yoy", "transactions",
                                     "housing_stock", "starts_uk", "comp_uk",
                                     "net_additions_uk", "turnover_rate"], errors="ignore")

print(f"Quarterly master: {df_master.shape}")
print(f"\nNon-null counts:")
print(df_master.notna().sum())

Quarterly master: (191, 14)

Non-null counts:
date                         191
year                         191
quarter                      191
starts_all                   191
starts_private               191
comp_all                     191
comp_private                 191
hpi_index                    191
hpi_price                    191
base_rate                    191
transactions_sa              154
transactions_spliced_flag    154
housing_stock_eng            185
epc_lodgements                68
dtype: int64


In [60]:
# Patch 2005 Q1 transactions gap by linear interpolation
df_master["transactions_sa"] = df_master["transactions_sa"].interpolate(method="linear")
df_master["transactions_spliced_flag"] = df_master["transactions_spliced_flag"].fillna(1).astype(int)

# Saving master datasets
df_master.to_csv("../data/processed/quarterly_master.csv", index=False)
df_net.to_csv("../data/processed/annual_net_additions.csv", index=False)

print(f"Saved quarterly_master.csv: {df_master.shape}")
print(f"Saved annual_net_additions.csv: {df_net.shape}")

Saved quarterly_master.csv: (191, 14)
Saved annual_net_additions.csv: (19, 9)


# NEED CONSTRUCTION COSTS